# Checkpointers (Part 3) — Combining Middlewares with Checkpointers

This notebook extends the [Middlewares (Part 3)](./Middlewares%20%28Part%203%29.ipynb) example by adding a **checkpointer** alongside the custom middlewares.

## What this demonstrates

- **Middlewares + Checkpointers together** — Both can be used on the same agent simultaneously.
- **State persistence for custom fields** — Not only conversation messages are checkpointed, but also the custom state fields defined by our middlewares (token counts, model call counts, TODO list).
- **Corrected `track_usage` implementation** — This version fixes a logic bug present in Middlewares (Part 3): the condition is now `is not None` (correct) instead of `is None` (wrong).
- **Thread-ID rule** — `thread_id` MUST be a string. Passing an integer silently breaks checkpoint lookup.

## Key difference from Checkpointers (Part 1) and (Part 2)

In Parts 1 & 2, only message history was checkpointed. Here, the agent state ALSO includes the custom fields from our middleware state classes (`model_calls`, `input_tokens`, etc.). All of it is persisted to the checkpointer.

In [ ]:
# Install required packages.
!pip install -q langchain langchain-openai

In [ ]:
import operator  # Provides operator.add for accumulating values in agent state

from google.colab import userdata
from langchain.agents import AgentState, create_agent
from langchain.agents.middleware import after_model
from langchain.agents.middleware import TodoListMiddleware
from langchain.messages import HumanMessage
from langchain_core.messages import BaseMessage
from langchain_core.runnables.config import RunnableConfig  # Type for the config dict (holds thread_id)
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.base import BaseCheckpointSaver   # Abstract base class for checkpointers
from langgraph.checkpoint.memory import InMemorySaver        # RAM-based checkpointer
from langgraph.runtime import Runtime
from typing import Annotated, List

openai_api_key = userdata.get('OPENAI_API_KEY')

def print_conversation(conversation: List[BaseMessage]):
    for message in conversation:
        message.pretty_print()

# Helper to list all checkpoints stored for a given thread.
def explore_checkpoints(checkpoint_saver: BaseCheckpointSaver, runnable_config: RunnableConfig):
    for checkpoint_tuple in checkpoint_saver.list(runnable_config):
        print(checkpoint_tuple)
        print(checkpoint_tuple.checkpoint["channel_values"])

In [ ]:
# --- Middleware 1: Track Token Usage (corrected version) ---
# This is the fixed implementation of track_usage from Middlewares (Part 3).
class TrackUsageState(AgentState):
    input_tokens: Annotated[int, operator.add]     # Running total of input tokens
    cached_tokens: Annotated[int, operator.add]    # Running total of cached tokens
    output_tokens: Annotated[int, operator.add]    # Running total of output tokens
    reasoning_tokens: Annotated[int, operator.add] # Running total of reasoning tokens


@after_model(state_schema=TrackUsageState)
def track_usage(state: TrackUsageState, runtime: Runtime):
    last_message = state["messages"][-1]

    # BUG FIX: The condition is now `is None` to early-exit when there's no metadata.
    # (Middlewares Part 3 had the condition inverted, causing it to never extract tokens.)
    if last_message.usage_metadata is None:
        return None  # Nothing to track — skip this model call

    # Start building the state update dict with the mandatory token fields.
    state_update = {
        "input_tokens": last_message.usage_metadata["input_tokens"],
        "output_tokens": last_message.usage_metadata["output_tokens"]
    }

    # Cache read and reasoning tokens are optional — only some models return them.
    if "input_token_details" in last_message.usage_metadata:
        state_update["cached_tokens"] = last_message.usage_metadata["input_token_details"]["cache_read"]
    if "output_token_details" in last_message.usage_metadata is not None:
        state_update["reasoning_tokens"] = last_message.usage_metadata["output_token_details"]["reasoning"]

    return state_update

In [ ]:
# --- Middleware 2: Count Model Calls ---
# Same as in Middlewares (Part 3). Increments a counter by 1 after each model call.
class CountModelCallsState(AgentState):
    model_calls: Annotated[int, operator.add]


@after_model(state_schema=CountModelCallsState)
def count_model_calls(state: CountModelCallsState, runtime: Runtime):
    return { "model_calls": 1 }

In [ ]:
# Create the in-memory checkpointer.
# This will persist all agent state (including the custom middleware fields) across messages.
checkpointer = InMemorySaver()

In [ ]:
# Build the agent with BOTH middlewares AND the checkpointer.
# This is the key combination of this notebook:
#   - Middlewares extend the state with custom fields (token counts, call counts, TODO list)
#   - Checkpointer persists the entire state (including those custom fields) across invocations
agent = create_agent(
    model=ChatOpenAI(model="gpt-5-nano", api_key=openai_api_key, reasoning_effort="low"),
    middleware=[track_usage, count_model_calls, TodoListMiddleware()],
    checkpointer=checkpointer
)

In [ ]:
# NOTE: The `thread_id` MUST be a string. Passing an integer (e.g. 1) instead of "1"
# will cause checkpoints to be stored under a different key, so you won't find them later.
config = {
    "configurable": {
        "thread_id": "1"  # Always use a string here!
    }
}

In [ ]:
# NOTE: When using checkpointers, you MUST always provide the config with thread_id.
# Without it, LangGraph doesn't know which thread to save to and will raise an error.
prove_irrational = agent.invoke(
    input={
        "messages": [
            HumanMessage("Prove that the square roots of 2 and 3 are irrational. First plan your actions, prepare a list of TODO items and then start working them one by one. I want you to provide at least 2 independent proofs for both tasks.")
        ]
    },
    config=config  # Required when using a checkpointer
)

In [ ]:
# Print the conversation with all reasoning steps visible.
print_conversation(prove_irrational['messages'])

In [ ]:
# Print the token usage and model call statistics.
# These custom state fields are ALSO persisted in the checkpointer
# alongside the standard conversation messages.
print(f"Model calls: {prove_irrational['model_calls']}")
print(f"Input tokens: {prove_irrational['input_tokens']}; Cache read: {prove_irrational['cached_tokens']}")
print(f"Output tokens: {prove_irrational['output_tokens']}; Reasoning: {prove_irrational['reasoning_tokens']}")

In [ ]:
# Inspect the checkpoints. Notice that the custom middleware fields
# (model_calls, input_tokens, etc.) appear in the saved channel_values —
# not just the messages. Everything that is part of the state gets checkpointed.
explore_checkpoints(checkpointer, config)